# Vision Transformer for Semantic Product Image Clustering - Basic Usage

This notebook demonstrates how to use the trained Vision Transformer model for embedding and clustering product images.

## Setup and Installation

First, let's install the package if not already installed:

In [ ]:
# Uncomment to install directly from GitHub
# !pip install git+https://github.com/polarSearch/polarsearch-vit.git

# Or install from local directory
# !pip install -e ..

## Import Dependencies

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import requests
from io import BytesIO
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
import os
import torchvision.transforms as transforms
import seaborn as sns

# Import our ViT package
from vit_product_clustering import ViTContrastive, get_transforms

## Load Pre-trained Model

Specify the path to your trained model weights:

In [ ]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Initialize model
model = ViTContrastive().to(device)

# Load weights
model_path = "path/to/your/vit_supcon_best.pth"  # Replace with your model path
try:
    model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"Model loaded successfully from {model_path}")
except FileNotFoundError:
    print(f"Model file not found: {model_path}")
    print("This example will continue using the untrained model for demonstration purposes.")

# Set model to evaluation mode
model.eval()

## Utility Functions for Processing Images

In [ ]:
# Define transforms
transform = get_transforms()

def load_image_from_url(url):
    """Load an image from a URL"""
    try:
        response = requests.get(url, timeout=5)
        img = Image.open(BytesIO(response.content)).convert('RGB')
        return img
    except Exception as e:
        print(f"Error loading image from {url}: {e}")
        # Return a blank image in case of error
        return Image.new('RGB', (224, 224), color='white')

def get_embedding(img, normalize=True):
    """Get embedding for a single image"""
    # Ensure image is in the right format
    if isinstance(img, str):
        # If input is a URL
        img = load_image_from_url(img)
    elif isinstance(img, np.ndarray):
        # If input is a numpy array
        img = Image.fromarray(img.astype('uint8'))
    
    # Apply transforms
    img_tensor = transform(img).unsqueeze(0).to(device)
    
    # Get embedding
    with torch.no_grad():
        embedding = model(img_tensor)
        if normalize:
            embedding = torch.nn.functional.normalize(embedding, dim=1)
    
    return embedding.cpu().numpy()[0]

def get_batch_embeddings(images, normalize=True):
    """Get embeddings for a batch of images"""
    # Process images
    processed_images = []
    for img in images:
        if isinstance(img, str):
            img = load_image_from_url(img)
        elif isinstance(img, np.ndarray):
            img = Image.fromarray(img.astype('uint8'))
        
        processed_images.append(transform(img))
    
    # Stack into a batch tensor
    batch_tensor = torch.stack(processed_images).to(device)
    
    # Get embeddings
    with torch.no_grad():
        embeddings = model(batch_tensor)
        if normalize:
            embeddings = torch.nn.functional.normalize(embeddings, dim=1)
    
    return embeddings.cpu().numpy()

## Example: Image Similarity Comparison

Let's demonstrate how to compare images for similarity:

In [ ]:
def cosine_similarity(emb1, emb2):
    """Calculate cosine similarity between two embeddings"""
    return np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))

def compare_images(image1, image2, threshold=0.7):
    """Compare two images and determine if they are similar"""
    emb1 = get_embedding(image1)
    emb2 = get_embedding(image2)
    
    similarity = cosine_similarity(emb1, emb2)
    
    print(f"Similarity score: {similarity:.4f}")
    if similarity > threshold:
        print("These images are semantically similar!")
    else:
        print("These images are different.")
    
    return similarity

# Example URLs of products to compare
# Replace with your own image URLs
image1_url = "https://example.com/product1.jpg"
image2_url = "https://example.com/product2.jpg"

# Compare the images
try:
    similarity = compare_images(image1_url, image2_url)
    
    # Visualize the images
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].imshow(load_image_from_url(image1_url))
    ax[0].set_title("Image 1")
    ax[0].axis('off')
    
    ax[1].imshow(load_image_from_url(image2_url))
    ax[1].set_title("Image 2")
    ax[1].axis('off')
    
    plt.suptitle(f"Similarity: {similarity:.4f}")
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"Error comparing images: {e}")

## Example: Clustering a Set of Images

Now let's demonstrate how to cluster a set of product images:

In [ ]:
# Example list of product image URLs
# Replace with your own image URLs
product_urls = [
    "https://example.com/product1.jpg",
    "https://example.com/product2.jpg",
    "https://example.com/product3.jpg",
    # Add more URLs...
]

# Number of clusters
n_clusters = 3

try:
    # Load images
    images = [load_image_from_url(url) for url in product_urls]
    
    # Get embeddings
    embeddings = get_batch_embeddings(images)
    
    # Cluster embeddings
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    clusters = kmeans.fit_predict(embeddings)
    
    # Visualize clusters using t-SNE
    tsne = TSNE(n_components=2, random_state=42)
    embeddings_2d = tsne.fit_transform(embeddings)
    
    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], c=clusters, cmap='viridis', s=100, alpha=0.8)
    plt.colorbar(scatter, label='Cluster')
    plt.title('t-SNE visualization of product image clusters')
    plt.xlabel('t-SNE dimension 1')
    plt.ylabel('t-SNE dimension 2')
    plt.tight_layout()
    plt.show()
    
    # Display images by cluster
    for cluster_id in range(n_clusters):
        cluster_indices = np.where(clusters == cluster_id)[0]
        n_images = len(cluster_indices)
        
        if n_images > 0:
            print(f"\nCluster {cluster_id} - {n_images} images:")
            
            # Display up to 5 images from this cluster
            n_display = min(5, n_images)
            fig, ax = plt.subplots(1, n_display, figsize=(15, 3))
            
            for i in range(n_display):
                if n_display == 1:
                    ax.imshow(images[cluster_indices[i]])
                    ax.set_title(f"Image {cluster_indices[i]}")
                    ax.axis('off')
                else:
                    ax[i].imshow(images[cluster_indices[i]])
                    ax[i].set_title(f"Image {cluster_indices[i]}")
                    ax[i].axis('off')
            
            plt.suptitle(f"Cluster {cluster_id}")
            plt.tight_layout()
            plt.show()
            
except Exception as e:
    print(f"Error clustering images: {e}")

## Example: Finding Similar Products

Now let's implement a function to find similar products to a query image:

In [ ]:
def find_similar_products(query_image, candidate_images, top_n=5):
    """Find the most similar products to a query image"""
    # Get embedding for query image
    query_emb = get_embedding(query_image)
    
    # Get embeddings for all candidate images
    candidate_embs = get_batch_embeddings(candidate_images)
    
    # Calculate similarities
    similarities = []
    for i, emb in enumerate(candidate_embs):
        sim = cosine_similarity(query_emb, emb)
        similarities.append((i, sim))
    
    # Sort by similarity (descending)
    similarities.sort(key=lambda x: x[1], reverse=True)
    
    # Return top N matches
    return similarities[:top_n]

# Example query image URL
query_url = "https://example.com/query_product.jpg"

try:
    # Find similar products
    similar_products = find_similar_products(query_url, product_urls)
    
    # Display results
    print("Query image:")
    plt.figure(figsize=(5, 5))
    plt.imshow(load_image_from_url(query_url))
    plt.title("Query Image")
    plt.axis('off')
    plt.show()
    
    print("\nSimilar products:")
    fig, ax = plt.subplots(1, len(similar_products), figsize=(15, 5))
    
    for i, (idx, similarity) in enumerate(similar_products):
        if len(similar_products) == 1:
            ax.imshow(images[idx])
            ax.set_title(f"Similarity: {similarity:.4f}")
            ax.axis('off')
        else:
            ax[i].imshow(images[idx])
            ax[i].set_title(f"Similarity: {similarity:.4f}")
            ax[i].axis('off')
    
    plt.suptitle("Similar Products")
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"Error finding similar products: {e}")

## Conclusion

This notebook has demonstrated several practical applications of the Vision Transformer model for product image analysis:

1. Comparing pairs of images for similarity
2. Clustering a collection of product images
3. Finding similar products to a query image

These capabilities can be integrated into e-commerce platforms for improved product organization, recommendation systems, and search functionality.